# 03 — R1, R3, R4: the challenger and what each constraint costs

**ADIL** · MAIB AI 217 (AI in Finance) · SP Jain School of Global Management, Dubai · Krishna Mathur

This is where the research question gets its first real answer. R1 gives LightGBM its best
shot — every candidate feature, no shape constraint — and then each rung adds one regulatory
constraint and the metrics move by exactly what that constraint cost.

| Rung | What it adds |
|---|---|
| R1 | LightGBM, unconstrained, full candidate pool |
| R3 | Monotone constraints on directionally-agreed features |
| R4 | Monotone, plus the scorecard's feature budget |

R2 is not a rung. Calibration is applied to every rung and to the scorecard alike, because a
miscalibrated model cannot support a cost-based threshold at all — it is a precondition of
notebook 06, not a constraint the ladder imposes.

Two things are checked rather than assumed: that the number of boosting rounds is chosen
without either model seeing more data than the other, and that the monotone constraints
**actually bind**. A declared constraint LightGBM quietly ignored would make R3 a rung that
costs nothing because it does nothing.

Outputs `metrics/r1.json`, `metrics/r3.json`, `metrics/r4.json`, `reports/challenger.md`, and
the predictions notebooks 04 to 06 read.

In [ ]:
import json
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression

from adil import challenger, constraints, evaluation, paths
from adil import scorecard as sc
from adil import split as sp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)

processed = paths.processed_dir()
frame = pd.read_parquet(processed / "adil_frame.parquet")
splits = pd.read_parquet(processed / "split_index.parquet")["split"].values
r0 = json.loads((paths.metrics_dir() / "r0.json").read_text())

features = sc.candidate_features(frame)
y = frame["TARGET"].values
is_train = splits == "train"
is_calibration = splits == "calibration"
is_test = splits == "test"

rungs = {rung.name: rung for rung in challenger.RUNGS}

# The feature cap only makes the comparison even if the scorecard actually used its
# whole budget. If IV selection had stopped short, capping the challenger at the
# declared number would hand it more features than the baseline had.
assert r0["n_characteristics"] == rungs["R4"].feature_cap, (
    f"scorecard used {r0['n_characteristics']} characteristics but R4's cap is "
    f"{rungs['R4'].feature_cap}; the feature-budget comparison would be uneven"
)

print(
    f"features: {len(features)}   train {is_train.sum():,}  "
    f"calibration {is_calibration.sum():,}  test {is_test.sum():,}"
)
print(f"R0 used {r0['n_characteristics']} characteristics; R4 capped at {rungs['R4'].feature_cap}")

## 1. The monotone declaration

`adil.constraints` was written and committed before this notebook ran. The rule for inclusion
is narrow on purpose: a direction goes in only if a credit officer would state it *without
looking at the data*. Directions that are merely empirically true in Home Credit are left out,
because a constraint justified by the training data is not a constraint, it is a fit.

The obvious temptation this guards against: fit, see which constraints hurt, quietly drop
those, report a smaller cost.

In [ ]:
constraint_table = constraints.constraint_table(features)
monotone_vector = constraint_table["constraint"].tolist()

print(
    f"{int((constraint_table['constraint'] != 0).sum())} of {len(features)} features "
    f"carry a declared direction ({(constraint_table['constraint'] != 0).mean():.1%})"
)
constraint_table.loc[constraint_table["constraint"] != 0].groupby(
    ["matched_pattern", "constraint"]
).size().rename("features").reset_index()

In [ ]:
for direction in constraints.DIRECTIONS:
    sign = "+1  risk rises with it" if direction.sign == 1 else "-1  risk falls with it"
    print(f"{direction.pattern}\n    {sign}\n    {direction.rationale}\n")

## 2. Fit the ladder

Boosting rounds are chosen by 5-fold cross-validation **within the training split**, then the
model is refitted on all of it. Early stopping against a held-out slice would have been
simpler and would have quietly handicapped the challenger: the scorecard saw all 184,506
training rows, so a challenger fitted on 80% of them would be losing to the split rather than
to the model class. The calibration and test splits are touched by neither model.

In [ ]:
design_train = challenger.design_matrix(frame.loc[is_train], features)
models, rounds, rung_features, timings = {}, {}, {}, {}

for name in ("R1", "R3"):
    rung = rungs[name]
    monotone = monotone_vector if rung.monotone else None
    started = time.time()
    rounds[name] = challenger.best_rounds(design_train, y[is_train], monotone=monotone)
    models[name] = challenger.fit(
        frame.loc[is_train],
        y[is_train],
        features,
        monotone=monotone,
        num_boost_round=rounds[name],
    )
    rung_features[name] = features
    timings[name] = time.time() - started
    print(f"{name}: {rounds[name]} rounds, {len(features)} features, {timings[name]:.0f}s")

In [ ]:
# R4 keeps the highest-gain features from R3, so the cap is a selection exercise
# rather than an arbitrary truncation, and the selection is made on training data only.
gains = challenger.gain_importance(models["R3"], features)
capped = gains.index[: rungs["R4"].feature_cap].tolist()
capped_constraints = constraints.monotone_constraints(capped)

started = time.time()
rounds["R4"] = challenger.best_rounds(
    challenger.design_matrix(frame.loc[is_train], capped),
    y[is_train],
    monotone=capped_constraints,
)
models["R4"] = challenger.fit(
    frame.loc[is_train],
    y[is_train],
    capped,
    monotone=capped_constraints,
    num_boost_round=rounds["R4"],
)
rung_features["R4"] = capped
timings["R4"] = time.time() - started
print(f"R4: {rounds['R4']} rounds, {len(capped)} features, {timings['R4']:.0f}s")
print(f"   of which constrained: {sum(1 for c in capped_constraints if c != 0)}")
print(
    f"   shared with the scorecard: "
    f"{len(set(capped) & set(r0['characteristics']))} of {len(capped)}"
)
pd.Series(gains.head(15), name="gain").to_frame()

## 3. Verification — the constraints actually bind

A declared constraint that LightGBM ignored would make R3 a rung costing nothing because it
does nothing, and the reported cost of monotonicity would be fiction.

The check sweeps **every** constrained feature across its observed range, with all other
features held at a reference applicant, and confirms the predicted probability moves only in
the declared direction. R1, fitted on the same data without the constraint, is swept alongside
as a control.

The control is the point. If R1 already happened to be monotone in a feature, the constraint
was never binding there and that feature says nothing about what monotonicity cost. Sweeping
all 42 rather than a chosen few is what turns "R3 is free" from a claim into a measurement:
free because the constraints did nothing would be a very different finding from free because
the data was already well behaved in the constrained directions.

Five reference applicants rather than one, so the conclusion does not rest on where a single
arbitrary row happened to sit.

In [ ]:
REFERENCE_ROWS = 5
GRID = 40
TOLERANCE = 1e-12

test_index = np.flatnonzero(is_test)
reference_index = test_index[np.linspace(0, len(test_index) - 1, REFERENCE_ROWS).astype(int)]


def worst_violation(model, model_features, feature, declared):
    # Largest step in the forbidden direction, across the reference applicants.
    values = frame.loc[is_test, feature]
    low, high = values.quantile(0.01), values.quantile(0.99)
    if not np.isfinite(low) or not np.isfinite(high) or low == high:
        return np.nan
    grid = np.linspace(low, high, GRID)
    worst = 0.0
    for position in reference_index:
        reference = frame.loc[[frame.index[position]], model_features]
        probe = pd.concat([reference] * GRID, ignore_index=True)
        probe[feature] = grid
        steps = np.diff(model.predict(challenger.design_matrix(probe, model_features)))
        violation = -steps.min() if declared == 1 else steps.max()
        worst = max(worst, float(violation))
    return worst


constrained = constraint_table.loc[constraint_table["constraint"] != 0]
checks = []
for row in constrained.itertuples():
    record = {"feature": row.feature, "declared": int(row.constraint)}
    for name in ("R1", "R3"):
        worst = worst_violation(models[name], rung_features[name], row.feature, row.constraint)
        record[name + " worst violation"] = worst
        record[name + " obeys"] = bool(np.isnan(worst) or worst <= TOLERANCE)
    checks.append(record)

monotonicity = pd.DataFrame(checks)
print(
    f"swept {len(monotonicity)} constrained features x {REFERENCE_ROWS} reference "
    f"applicants x {GRID} grid points, under R1 and R3"
)
monotonicity.head(12)

In [ ]:
binding = monotonicity[~monotonicity["R1 obeys"] & monotonicity["R3 obeys"]]
never_bound = monotonicity[monotonicity["R1 obeys"]]

print(f"R3 obeys every declared direction : {bool(monotonicity['R3 obeys'].all())}")
print(f"constraints that changed the shape: {len(binding)} of {len(monotonicity)}")
print(f"constraints that never bound      : {len(never_bound)} of {len(monotonicity)}")
print(f"largest violation by R1           : {monotonicity['R1 worst violation'].max():.3e}")
print(f"largest violation by R3           : {monotonicity['R3 worst violation'].max():.3e}")
print("")
print("features where R1 violates its declared direction and R3 does not:")
for feature in binding["feature"]:
    print(f"  {feature}")

assert monotonicity["R3 obeys"].all(), "a declared constraint did not bind in R3"
assert len(binding) > 0, (
    "no constraint changed anything, so R3's cost is not attributable to monotonicity"
)

## 4. Calibration

Isotonic regression fitted on the calibration split, which no model has seen. Applied to every
rung on the same terms the scorecard got in notebook 02.

In [ ]:
probability, calibrated, isotonics = {}, {}, {}
for name, model in models.items():
    columns = rung_features[name]
    probability[name] = {
        part: model.predict(challenger.design_matrix(frame.loc[mask], columns))
        for part, mask in [("train", is_train), ("calibration", is_calibration), ("test", is_test)]
    }
    isotonics[name] = IsotonicRegression(out_of_bounds="clip").fit(
        probability[name]["calibration"], y[is_calibration]
    )
    calibrated[name] = {
        part: isotonics[name].predict(values) for part, values in probability[name].items()
    }
print("calibrated:", list(calibrated))

## 5. The ladder

In [ ]:
rows = []
r0_raw = r0["metrics_by_split"]["test"]
rows.append({**r0_raw, "rung": "R0", "state": "raw", "features": r0["n_characteristics"]})
rows.append(
    {
        **r0["calibrated_test"],
        "rung": "R0",
        "state": "calibrated",
        "features": r0["n_characteristics"],
    }
)
for name in ("R1", "R3", "R4"):
    rows.append(
        {
            **evaluation.metric_set(y[is_test], probability[name]["test"], split="test"),
            "rung": name,
            "state": "raw",
            "features": len(rung_features[name]),
        }
    )
    rows.append(
        {
            **evaluation.metric_set(y[is_test], calibrated[name]["test"], split="test"),
            "rung": name,
            "state": "calibrated",
            "features": len(rung_features[name]),
        }
    )

ladder = pd.DataFrame(rows).set_index(["rung", "state"])
ladder = ladder[["features", "pr_auc", "auc", "ks", "gini", "brier", "ece"]]
ladder.round(5)

In [ ]:
calibrated_only = ladder.xs("calibrated", level="state")
deltas = calibrated_only.diff()
deltas.insert(0, "vs", ["—", "R1 - R0", "R3 - R1", "R4 - R3"])
print("What each step of the ladder cost, on calibrated test predictions:")
deltas.round(5)

## 6. Overfitting check

The challenger has 567 features and the scorecard has 20, so the train-to-test gap is the
thing to look at before believing any of the gain.

In [ ]:
gaps = []
for name in ("R1", "R3", "R4"):
    train_metrics = evaluation.metric_set(y[is_train], probability[name]["train"], split="train")
    test_metrics = evaluation.metric_set(y[is_test], probability[name]["test"], split="test")
    gaps.append(
        {
            "rung": name,
            "train AUC": train_metrics["auc"],
            "test AUC": test_metrics["auc"],
            "gap": train_metrics["auc"] - test_metrics["auc"],
            "rounds": rounds[name],
        }
    )
gaps.append(
    {
        "rung": "R0",
        "train AUC": r0["metrics_by_split"]["train"]["auc"],
        "test AUC": r0["metrics_by_split"]["test"]["auc"],
        "gap": r0["metrics_by_split"]["train"]["auc"] - r0["metrics_by_split"]["test"]["auc"],
        "rounds": np.nan,
    }
)
pd.DataFrame(gaps).set_index("rung").round(5)

## 7. Test-set intervals

The gain between two rungs is only a finding if it is larger than the uncertainty in
measuring it.

In [ ]:
def pr_auc_of(truth, prob):
    return float(evaluation.metric_set(truth, prob, split="test")["pr_auc"])


interval_rows = []
for name in ("R1", "R3", "R4"):
    low, high = evaluation.bootstrap_interval(y[is_test], calibrated[name]["test"], pr_auc_of)
    interval_rows.append(
        {
            "rung": name,
            "pr_auc": float(calibrated_only.loc[name, "pr_auc"]),
            "low": low,
            "high": high,
        }
    )
interval_rows.append(
    {
        "rung": "R0",
        "pr_auc": float(calibrated_only.loc["R0", "pr_auc"]),
        "low": r0["test_intervals"]["pr_auc"]["low"],
        "high": r0["test_intervals"]["pr_auc"]["high"],
    }
)
intervals = pd.DataFrame(interval_rows).set_index("rung").loc[["R0", "R1", "R3", "R4"]]
intervals.round(5)

## 8. Persist

In [ ]:
predictions = pd.DataFrame(
    {
        "SK_ID_CURR": frame["SK_ID_CURR"].values,
        "split": splits,
        "TARGET": y,
    }
)
for name in ("R1", "R3", "R4"):
    raw_column = np.empty(len(frame), dtype=float)
    calibrated_column = np.empty(len(frame), dtype=float)
    for part, mask in [("train", is_train), ("calibration", is_calibration), ("test", is_test)]:
        raw_column[mask] = probability[name][part]
        calibrated_column[mask] = calibrated[name][part]
    predictions[f"{name}_prob"] = raw_column
    predictions[f"{name}_prob_calibrated"] = calibrated_column
    assert np.allclose(predictions.loc[is_test, f"{name}_prob"], probability[name]["test"])

# The fitted boosters are persisted so notebook 04 explains the model that produced
# these numbers rather than a refit of it. LightGBM's text format is version-portable
# and diffable, which a pickle is not.
for name, model in models.items():
    model.save_model(str(processed / f"{name.lower()}_model.txt"))
    pd.Series(rung_features[name], name="feature").to_frame().to_parquet(
        processed / f"{name.lower()}_features.parquet", index=False
    )

predictions.to_parquet(processed / "challenger_predictions.parquet", index=False)
constraint_table.to_parquet(processed / "constraint_table.parquet", index=False)
gains.rename("gain").to_frame().to_parquet(processed / "r3_gain_importance.parquet")
print(f"wrote challenger_predictions.parquet ({predictions.shape[1]} columns)")

In [ ]:
for name in ("R1", "R3", "R4"):
    rung = rungs[name]
    payload = {
        "rung": name,
        "description": rung.description,
        "seed": sp.SEED,
        "monotone": rung.monotone,
        "feature_cap": rung.feature_cap,
        "n_features": len(rung_features[name]),
        "n_constrained_features": int(
            sum(1 for c in constraints.monotone_constraints(rung_features[name]) if c != 0)
        ),
        "boosting_rounds": rounds[name],
        "params": {k: v for k, v in challenger.BASE_PARAMS.items() if k != "num_threads"},
        "metrics_test_raw": evaluation.metric_set(
            y[is_test], probability[name]["test"], split="test"
        ),
        "metrics_test_calibrated": evaluation.metric_set(
            y[is_test], calibrated[name]["test"], split="test"
        ),
        "metrics_train_raw": evaluation.metric_set(
            y[is_train], probability[name]["train"], split="train"
        ),
        "pr_auc_interval": {
            "low": float(intervals.loc[name, "low"]),
            "high": float(intervals.loc[name, "high"]),
        },
    }
    if name == "R4":
        payload["features"] = rung_features[name]
        payload["shared_with_scorecard"] = sorted(
            set(rung_features[name]) & set(r0["characteristics"])
        )
    if name == "R3":
        payload["monotonicity_check"] = monotonicity.to_dict("records")
    (paths.metrics_dir() / f"{name.lower()}.json").write_text(
        json.dumps(payload, indent=2, default=float) + "\n"
    )
    print(f"wrote metrics/{name.lower()}.json")

In [ ]:
r1c = calibrated_only.loc["R1"]
r0c = calibrated_only.loc["R0"]
r3c = calibrated_only.loc["R3"]
r4c = calibrated_only.loc["R4"]
shared = len(set(rung_features["R4"]) & set(r0["characteristics"]))
constrained_count = int((constraint_table["constraint"] != 0).sum())
binding_names = ", ".join(f"`{n}`" for n in binding["feature"]) or "none"

lines = [
    "# ADIL — R1, R3, R4: the challenger and what each constraint costs",
    "",
    "Generated by `notebooks/03_challenger.ipynb`. Every number is read from a fitted",
    "model, not typed.",
    "",
    "MAIB AI 217 · SP Jain School of Global Management, Dubai · Krishna Mathur",
    "",
    "## Setup",
    "",
    f"- Candidate features: **{len(features)}**",
    f"- Carrying a declared monotone direction: **{constrained_count}** "
    f"({constrained_count / len(features):.1%})",
    f"- Train / calibration / test: {is_train.sum():,} / {is_calibration.sum():,} / "
    f"{is_test.sum():,}",
    f"- Seed: {sp.SEED}",
    "",
    "Boosting rounds are chosen by 5-fold cross-validation **within the training split**,",
    "then each model is refitted on all of it. Early stopping against a held-out slice",
    "would have been simpler and would have handicapped the challenger: the scorecard saw",
    "all training rows, so a challenger fitted on 80% of them would lose to the split",
    "rather than to the model class. Neither model touches calibration or test.",
    "",
    "`adil.constraints` was committed before this notebook ran. A direction is declared",
    "only where a credit officer would state it without looking at the data; directions",
    "that are merely empirically true in Home Credit are left out, because a constraint",
    "justified by the training data is a fit, not a constraint.",
    "",
    "## Verification — the constraints bind",
    "",
    "A declared constraint LightGBM ignored would make R3 a rung that costs nothing",
    "because it does nothing. All",
    f"{len(monotonicity)} constrained features were swept across their observed range at",
    f"{REFERENCE_ROWS} reference applicants and {GRID} grid points, under R3 and under the",
    "unconstrained R1 as a control.",
    "",
    "| | R1 | R3 |",
    "|---|---:|---:|",
    f"| Features obeying their declared direction | "
    f"{int(monotonicity['R1 obeys'].sum())} / {len(monotonicity)} | "
    f"{int(monotonicity['R3 obeys'].sum())} / {len(monotonicity)} |",
    f"| Largest step in the forbidden direction | "
    f"{monotonicity['R1 worst violation'].max():.2e} | "
    f"{monotonicity['R3 worst violation'].max():.2e} |",
    "",
    f"R3 obeys every declared direction. On **{len(binding)} of {len(monotonicity)}**",
    "features the unconstrained model does not, and those are the features where the",
    "constraint actually changed the fitted shape:",
    "",
    f"{binding_names}",
    "",
    f"The remaining {len(never_bound)} constraints never bound — R1 was already monotone in",
    "those directions, so imposing the constraint cost nothing there because there was",
    "nothing to give up. This distinction is what makes the near-zero R3 - R1 step below a",
    "measurement rather than an artifact: monotonicity is cheap here because the data was",
    "largely well behaved in the declared directions, not because the constraint was inert.",
    "",
    "## The ladder",
    "",
    "Calibrated test predictions. PR-AUC leads; at an 8% base rate AUC flatters.",
    "",
    "| Rung | Features | PR-AUC | AUC | KS | Gini | Brier | ECE |",
    "|---|---:|---:|---:|---:|---:|---:|---:|",
]
for name in ["R0", "R1", "R3", "R4"]:
    row = calibrated_only.loc[name]
    lines.append(
        f"| {name} | {int(row['features'])} | {row['pr_auc']:.4f} | {row['auc']:.4f} | "
        f"{row['ks']:.4f} | {row['gini']:.4f} | {row['brier']:.5f} | {row['ece']:.5f} |"
    )
lines += [
    "",
    "### What each step cost",
    "",
    "| Step | PR-AUC | AUC | Gini | Brier |",
    "|---|---:|---:|---:|---:|",
    f"| R1 - R0 (model class) | {r1c['pr_auc'] - r0c['pr_auc']:+.4f} | "
    f"{r1c['auc'] - r0c['auc']:+.4f} | {r1c['gini'] - r0c['gini']:+.4f} | "
    f"{r1c['brier'] - r0c['brier']:+.5f} |",
    f"| R3 - R1 (monotonicity) | {r3c['pr_auc'] - r1c['pr_auc']:+.4f} | "
    f"{r3c['auc'] - r1c['auc']:+.4f} | {r3c['gini'] - r1c['gini']:+.4f} | "
    f"{r3c['brier'] - r1c['brier']:+.5f} |",
    f"| R4 - R3 (feature budget) | {r4c['pr_auc'] - r3c['pr_auc']:+.4f} | "
    f"{r4c['auc'] - r3c['auc']:+.4f} | {r4c['gini'] - r3c['gini']:+.4f} | "
    f"{r4c['brier'] - r3c['brier']:+.5f} |",
    f"| R4 - R0 (net) | {r4c['pr_auc'] - r0c['pr_auc']:+.4f} | "
    f"{r4c['auc'] - r0c['auc']:+.4f} | {r4c['gini'] - r0c['gini']:+.4f} | "
    f"{r4c['brier'] - r0c['brier']:+.5f} |",
    "",
    "### Test-set intervals on PR-AUC",
    "",
    "A step is a finding only if it is larger than the uncertainty in measuring it.",
    "Percentile bootstrap over resampled test predictions.",
    "",
    "| Rung | PR-AUC | 2.5% | 97.5% |",
    "|---|---:|---:|---:|",
]
for name in ["R0", "R1", "R3", "R4"]:
    row = intervals.loc[name]
    lines.append(f"| {name} | {row['pr_auc']:.4f} | {row['low']:.4f} | {row['high']:.4f} |")
lines += [
    "",
    "## Overfitting",
    "",
    "The challenger carries 567 features against the scorecard's 20, so the train-to-test",
    "gap is worth reading before believing any gain.",
    "",
    "| Rung | Train AUC | Test AUC | Gap | Rounds |",
    "|---|---:|---:|---:|---:|",
]
for row in gaps:
    rounds_text = "—" if not np.isfinite(row["rounds"]) else f"{int(row['rounds'])}"
    lines.append(
        f"| {row['rung']} | {row['train AUC']:.4f} | {row['test AUC']:.4f} | "
        f"{row['gap']:.4f} | {rounds_text} |"
    )
lines += [
    "",
    "## R4's features",
    "",
    f"R4 keeps the {len(rung_features['R4'])} highest-gain features from R3, selected on",
    f"training data only. **{shared} of them** are also scorecard characteristics, so the",
    "two models are not merely agreeing on the same columns by construction.",
    "",
    "| # | Feature | R3 gain |",
    "|---:|---|---:|",
]
for i, feature in enumerate(rung_features["R4"], start=1):
    mark = " *" if feature in r0["characteristics"] else ""
    lines.append(f"| {i} | `{feature}`{mark} | {gains[feature]:,.0f} |")
lines += [
    "",
    "`*` also a scorecard characteristic.",
    "",
    "## Limitations",
    "",
    "- Hyperparameters are fixed across rungs and deliberately untuned. This project asks",
    "  what constraints cost, not how high the AUC can be pushed; a tuned challenger would",
    "  confound the two. A tuned model would likely score higher and the ladder's *steps*",
    "  are what matter here, not its absolute level.",
    "- Intervals cover measurement on this test set. They do not cover how much a metric",
    "  would move on different training data, which is larger and is not estimated.",
    "- R4's features are chosen by R3's gain importance, so the cap inherits whatever",
    "  R3 happened to favour. A different selection rule would give a different R4.",
    "- No out-of-time validation is possible; see `reports/data_quality.md`.",
    "",
]
path = paths.reports_dir() / "challenger.md"
path.write_text("\n".join(lines) + "\n")
print(f"wrote {path} ({len(lines)} lines)")